In [1]:
import pandas as pd
import warnings 
import json
import os
warnings.filterwarnings('ignore')

In [2]:
city = 'Mumbai'

df = pd.read_json(rf'../web scraping/{city}/car_dataset_{city.lower()}.json', lines=True)
df.head()

,url,car_name,Price,Registration Year,Insurance,Fuel Type,Seats,Kms Driven,RTO,Ownership,...,"<a href=""/car-faqs/mg-zs-ev/what-is-the-battery-capacity-of-mg-zs-ev-203.html"" class=""bluelink"" title=""Battery Capacity"">Battery Capacity</a>",Wireless Charging,Charger Type,Charging Time (15 A Plug Point),Charging Time (7.2 kW AC Fast Charger),Charging Time (50 kW DC Fast Charger),Petrol Mileage (ARAI),Approach Angle,Break-over Angle,Departure Angle
0,https://www.cardekho.com/used-car-details/used...,Honda Amaze,₹5.40 Lakh,2020,-,Petrol,5 Seats,"10,000 Kms",Mumbai,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,https://www.cardekho.com/buy-used-car-details/...,Maruti Suzuki Vitara Brezza,₹6.49 Lakh,Dec 2020,Comprehensive,Petrol,5 Seats,"24,100 Kms",Mumbai,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,https://www.cardekho.com/used-car-details/used...,Skoda Kushaq,₹9.45 Lakh,Nov 2021,-,Petrol,5 Seats,"37,212 Kms",Mumbai,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,https://www.cardekho.com/used-car-details/used...,Volkswagen Ameo,₹3.55 Lakh,Nov 2016,Third Party,Petrol,5 Seats,"35,413 Kms",Mumbai,Second Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,https://www.cardekho.com/used-car-details/used...,Maruti Suzuki Ciaz,₹6.60 Lakh,2019,-,Petrol,5 Seats,"40,000 Kms",Mumbai,First Owner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [3]:
PROCESSED_FILES_LOG = 'processed_files.txt'

def get_processed_files():
    if os.path.exists(PROCESSED_FILES_LOG):
        with open(PROCESSED_FILES_LOG, 'r') as f:
            return f.read().splitlines()
    return []

def mark_file_as_processed(json_filename):
    processed = get_processed_files()
    if json_filename not in processed:
        with open(PROCESSED_FILES_LOG, 'a') as f:
            f.write(json_filename + '\n')

In [4]:
# ─────────────────────────────────────────
json_file = f'car_dataset_{city.lower()}.json'

if json_file in get_processed_files():
    print(f"⚠️ '{json_file}' already processed — skipping!")
    df = pd.read_csv('dataset.csv')  
    print(f"Loaded existing dataset with {len(df)} rows")
    
else:  
    req_col = []
    with open('features.txt', 'r') as f:
        features = f.read().split('\n')
    for i in features:
        if i in df.columns:
            req_col.append(i) 
    df = df[req_col]

    before = len(df)
    df = df.drop_duplicates()
    print(f"Duplicates removed from new data: {before - len(df)} rows")

    if os.path.exists('dataset.csv') and os.path.getsize('dataset.csv') > 0:
        existing_df = pd.read_csv('dataset.csv')
        combined_df = pd.concat([existing_df, df], ignore_index=True)

        before = len(combined_df)
        combined_df = combined_df.drop_duplicates()
        print(f"Duplicates removed from combined data: {before - len(combined_df)} rows")

        combined_df.to_csv('dataset.csv', index=False)
        print(f"Appended {len(df)} rows. Total rows: {len(combined_df)}")
    else:
        df.to_csv('dataset.csv', index=False)
        print(f"Created new dataset.csv with {len(df)} rows")

    mark_file_as_processed(json_file)
    print(f"✅ '{json_file}' marked as processed!")

⚠️ 'car_dataset_mumbai.json' already processed — skipping!
Loaded existing dataset with 3628 rows


In [5]:
df = pd.read_csv('dataset.csv')

In [6]:
df.head()

,Mileage,Engine,Kerb Weight,Fuel,Transmission Type,Power,No. of Cylinders,Registration Year
0,18.9 kmpl,1197 cc,935 kg,Petrol,Manual,82 bhp,4.0,2015
1,19.81 kmpl,1086 cc,860 kg,Petrol,Manual,68.05 bhp,4.0,Apr 2015
2,15.6 kmpl,1196 cc,1090 kg,Petrol,Manual,70 bhp,4.0,Dec 2019
3,18.9 kmpl,1197 cc,1060 kg,Petrol,Manual,81.86 bhp,4.0,Jul 2017
4,25.44 kmpl,936 cc,1025 kg,Diesel,Manual,56.3 bhp,3.0,2015


In [7]:
df.shape

(3628, 8)

In [8]:
# Missing percentage count
for col in df.columns:
    missing_count = df[col].isnull().sum()
    missing_percentage = (missing_count / len(df)) * 100
    print(f"{col}: {missing_percentage:.2f}% missing")

Mileage: 13.67% missing
Engine: 0.88% missing
Kerb Weight: 8.32% missing
Fuel: 12.76% missing
Transmission Type: 0.08% missing
Power: 2.32% missing
No. of Cylinders: 0.61% missing
Registration Year: 0.14% missing


In [9]:
def handle_missing_values(df, col):
    missing_pct = df[col].isnull().mean() * 100
    print(f"{col}: {missing_pct:.2f}% missing", end=" → ")

    # Drop if more than 50% missing
    if missing_pct > 50:
        df.drop(columns=[col], inplace=True)
        print("❌ Dropped (>50% missing)")

    # Between 30% and 50% — drop rows
    elif missing_pct > 30:
        df.dropna(subset=[col], inplace=True)
        print("🗑️ Dropped rows (30-50% missing)")

    # Between 10% and 30% — fill with mean/mode + add indicator column
    elif missing_pct > 10:
        if df[col].dtype == 'object':
            fill_val = df[col].mode()[0]
            strategy = f"mode ({fill_val})"
        else:
            fill_val = df[col].mean()
            strategy = f"mean ({fill_val:.2f})"

        df[f'{col}_was_missing'] = df[col].isnull().astype(int)  # indicator column
        df[col].fillna(fill_val, inplace=True)
        print(f"⚠️ Filled with {strategy} + added indicator column (10-30% missing)")

    # Less than 10% — fill with median/mode
    elif missing_pct > 0:
        if df[col].dtype == 'object':
            fill_val = df[col].mode()[0]
            strategy = f"mode ({fill_val})"
        else:
            fill_val = df[col].median()
            strategy = f"median ({fill_val:.2f})"

        df[col].fillna(fill_val, inplace=True)
        print(f"✅ Filled with {strategy} (<10% missing)")

    else:
        print("✅ No missing values")

    return df


# Apply to all columns
cols_with_missing = ['Mileage', 'Engine', 'Kerb Weight', 'Fuel',
                     'Transmission Type', 'Power', 'No. of Cylinders', 'Registration Year']

for col in cols_with_missing:
    df = handle_missing_values(df, col)

Mileage: 13.67% missing → ⚠️ Filled with mode (18.9 kmpl) + added indicator column (10-30% missing)
Engine: 0.88% missing → ✅ Filled with mode (1197 cc) (<10% missing)
Kerb Weight: 8.32% missing → ✅ Filled with mode (1100 kg) (<10% missing)
Fuel: 12.76% missing → ⚠️ Filled with mode (Petrol) + added indicator column (10-30% missing)
Transmission Type: 0.08% missing → ✅ Filled with mode (Manual) (<10% missing)
Power: 2.32% missing → ✅ Filled with mode (81.86 bhp) (<10% missing)
No. of Cylinders: 0.61% missing → ✅ Filled with median (4.00) (<10% missing)
Registration Year: 0.14% missing → ✅ Filled with mode (Jun 2022) (<10% missing)


In [10]:
df['Fuel'].value_counts()

Fuel
Petrol    2777
Diesel     783
CNG         68
Name: count, dtype: int64

1. Mileage

In [11]:
df['Mileage'] = df['Mileage'].str.extract(r'(\d+\.?\d*)').astype(float)

2. Engine

In [12]:
df['Engine'] = df['Engine'].str.extract(r'(\d+)').astype(float)

3. Weight

In [13]:
df['Kerb Weight'] = df['Kerb Weight'].str.extract(r'(\d+)').astype(float)

4. Power

In [14]:
df['Power'] = df['Power'].str.extract(r'(\d+\.?\d*)').astype(float)

5. Registration Year

In [15]:
df['Registration Year'] = pd.to_numeric(
    df['Registration Year'].str.extract(r'(\d{4})')[0],
    errors='coerce'
)

6. Transmission Type

In [16]:
df['Transmission Type'] = df['Transmission Type'].map({
                                'Automatic': 1,
                                'Manual': 0
                            })

7. Fuel

In [17]:
def encode_and_show_dropped(df, column, drop_first=True):
    dummies = pd.get_dummies(df[column], prefix=column, drop_first=drop_first)
    
    all_categories = df[column].unique()
    encoded_categories = [col.replace(f"{column}_", "") for col in dummies.columns]
    dropped_categories = [cat for cat in all_categories if cat not in encoded_categories]
    
    print(f"All categories:     {list(all_categories)}")
    print(f"Encoded categories: {encoded_categories}")
    print(f"Dropped categories: {dropped_categories}")
    
    return dummies

# usage
fuel_dummies = encode_and_show_dropped(df, 'Fuel', drop_first=True)
df = pd.concat([df.drop(columns=['Fuel']), fuel_dummies], axis=1)


All categories:     ['Petrol', 'Diesel', 'CNG']
Encoded categories: ['Diesel', 'Petrol']
Dropped categories: ['CNG']


In [18]:
df['Mileage'].fillna(df['Mileage'].median(), inplace=True)

In [19]:
df.columns

Index(['Mileage', 'Engine', 'Kerb Weight', 'Transmission Type', 'Power',
       'No. of Cylinders', 'Registration Year', 'Mileage_was_missing',
       'Fuel_was_missing', 'Fuel_Diesel', 'Fuel_Petrol'],
      dtype='object')

---

In [20]:
from sklearn.model_selection import train_test_split, GridSearchCV 
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

In [21]:
# Features and target
X = df.drop(columns=['Mileage'])
y = df['Mileage']

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [22]:
model = RandomForestRegressor()

# Hyperparameter grid
param_grid = {
    'n_estimators': [100, 200],        
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5]
}

# Grid search
grid_search = GridSearchCV(
    model,                            
    param_grid,
    cv=5,
    scoring='r2',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_model = grid_search.best_estimator_

print("Best parameters:", grid_search.best_params_)
print("Best CV score:", grid_search.best_score_)

y_pred = best_model.predict(X_test)
r2 = r2_score(y_test, y_pred)
print("R2 Score:", r2)

Best parameters: {'max_depth': None, 'min_samples_split': 2, 'n_estimators': 200}
Best CV score: 0.8976552241605615
R2 Score: 0.9463085487009394


In [23]:
import os
import joblib
os.makedirs("model", exist_ok=True)
joblib.dump(best_model, "model/mileage_model.pkl")
print("Model saved!")


Model saved!


In [24]:
X_train.head(1)

,Engine,Kerb Weight,Transmission Type,Power,No. of Cylinders,Registration Year,Mileage_was_missing,Fuel_was_missing,Fuel_Diesel,Fuel_Petrol
1462,1497.0,1305.0,0,108.5,4.0,2020,0,0,True,False
